In [2]:
from __future__ import annotations

import argparse
import csv
import pickle
import re
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import tifffile


# -----------------------------------------------------------------------------
# Data containers
# -----------------------------------------------------------------------------


@dataclass
class TimepointPaths:
    """Filesystem paths for one timepoint."""

    name: str
    folder: Path
    raw_path: Path
    segmentation_path: Path
    feature_paths: list[Path]
    ignored_feature_paths: list[Path] = field(default_factory=list)


@dataclass
class LabelGeometry:
    """Per-label geometry in internal Y-X-Z axis order."""

    label_id: int
    vox_coords: np.ndarray  # (N, 3), dtype int32, columns are [y, x, z]
    centroid: np.ndarray  # (3,), float64
    volume: int
    bbox_start: np.ndarray  # (3,), inclusive
    bbox_stop: np.ndarray  # (3,), exclusive
    local_mask: np.ndarray  # cropped binary mask in bbox coordinates
    local_centroid: np.ndarray  # centroid in bbox coordinates

    @property
    def bbox_slices(self) -> tuple[slice, slice, slice]:
        return tuple(
            slice(int(a), int(b)) for a, b in zip(self.bbox_start, self.bbox_stop)
        )


@dataclass
class PreparedTimepoint:
    """One timepoint with discovered paths and prepared segmentation geometry."""

    index: int  # 1-based timepoint index
    paths: TimepointPaths
    shape_yxz: tuple[int, int, int]
    geometries: dict[int, LabelGeometry]
    label_ids: np.ndarray
    centroids: np.ndarray  # (N, 3)
    volumes: np.ndarray  # (N,)


@dataclass
class CandidateVote:
    candidate_label: int
    wins: int = 0
    features: list[int] = field(default_factory=list)  # 0-based feature indices
    distance: float | None = None
    dice: float | None = None


@dataclass
class RefSummary:
    ref_label: int
    candidates: list[CandidateVote]


@dataclass
class AssignmentRecord:
    ref_label: int
    candidate_label: int
    method: str
    distance: float | None = None
    wins: int | None = None


@dataclass
class PairResult:
    t_ref: int
    t_cand: int
    initial_summary: list[RefSummary]
    summary_current: list[RefSummary]
    summary_history: list[list[RefSummary]]
    assignments: list[AssignmentRecord]
    final_assignment: dict[int, int]
    summary_distance_prefilter: list[dict[str, float]]


@dataclass
class TrackPoint:
    timepoint: int
    label_id: int
    centroid: tuple[float, float, float]
    volume: int


@dataclass
class Track:
    track_id: int
    points: list[TrackPoint]
    start_time: int
    length: int


@dataclass
class RefCandidateMetrics:
    """All sparse pairwise metrics for one reference label against its candidate pool."""

    ref_label: int
    candidate_ids: np.ndarray  # shape (C,), actual segmentation IDs
    distances: np.ndarray  # shape (C,), anisotropic centroid distances
    dice: np.ndarray  # shape (C,), shape-overlap score after centroid alignment
    overlap_counts: np.ndarray  # shape (C,), number of overlapping voxels after alignment
    corr: np.ndarray  # shape (n_features, C), NaN if invalid
    mse: np.ndarray  # shape (n_features, C), NaN if invalid


# -----------------------------------------------------------------------------
# Utility functions
# -----------------------------------------------------------------------------


def natural_sort_key(value: str) -> list[Any]:
    parts = re.split(r"(\d+)", value)
    key: list[Any] = []
    for part in parts:
        if part.isdigit():
            key.append(int(part))
        else:
            key.append(part.lower())
    return key


TIFF_SUFFIXES = {".tif", ".tiff"}


def list_tiff_files(folder: Path) -> list[Path]:
    return sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in TIFF_SUFFIXES],
        key=lambda p: natural_sort_key(p.name),
    )


class PipelineError(RuntimeError):
    pass


def _to_internal_yxz(volume: np.ndarray, axes: str) -> np.ndarray:
    axes = axes.lower()
    if volume.ndim != 3:
        raise PipelineError(f"Expected a 3D TIFF volume, got shape {volume.shape}.")
    if axes == "yxz":
        return volume
    if axes == "zyx":
        return np.moveaxis(volume, 0, -1)
    raise ValueError("axes must be either 'yxz' or 'zyx'.")


def _from_internal_yxz(volume_yxz: np.ndarray, axes: str) -> np.ndarray:
    axes = axes.lower()
    if axes == "yxz":
        return volume_yxz
    if axes == "zyx":
        return np.moveaxis(volume_yxz, -1, 0)
    raise ValueError("axes must be either 'yxz' or 'zyx'.")


def read_tiff_volume(path: str | Path, axes: str = "zyx") -> np.ndarray:
    arr = tifffile.imread(path)
    return _to_internal_yxz(np.asarray(arr), axes)


def write_tiff_volume(path: str | Path, volume_yxz: np.ndarray, axes: str = "zyx") -> None:
    arr = _from_internal_yxz(np.asarray(volume_yxz), axes)
    tifffile.imwrite(path, arr)


def discover_segmentation_file(
    timepoint_dir: Path,
    raw_filename: str,
    features_dirname: str,
    segmentation_filename: str | None,
) -> Path:
    if segmentation_filename is not None:
        seg_path = timepoint_dir / segmentation_filename
        if not seg_path.exists():
            raise PipelineError(f"Segmentation file not found: {seg_path}")
        return seg_path

    candidates = [
        p
        for p in list_tiff_files(timepoint_dir)
        if p.name != raw_filename and p.parent.name != features_dirname
    ]
    if len(candidates) == 1:
        return candidates[0]

    heuristic = [
        p
        for p in candidates
        if any(token in p.stem.lower() for token in ("seg", "instance", "label", "mask"))
    ]
    if len(heuristic) == 1:
        return heuristic[0]

    msg = [
        f"Could not determine the segmentation file in {timepoint_dir}.",
        f"Found candidates: {[p.name for p in candidates]}",
        "Pass segmentation_filename explicitly.",
    ]
    raise PipelineError(" ".join(msg))


# -----------------------------------------------------------------------------
# Experiment loading (beginning of MATLAB script 1, cleaned up)
# -----------------------------------------------------------------------------


def discover_experiment(
    experiment_dir: str | Path,
    *,
    raw_filename: str = "volume_unnorm.tif",
    segmentation_filename: str | None = None,
    features_dirname: str = "features_tif",
    n_patch_features: int = 384,
) -> list[TimepointPaths]:
    """
    Discover timepoints and files in the experiment folder.

    Timepoint folders are sorted alphabetically / naturally. Feature TIFFs are naturally
    sorted and truncated to the first `n_patch_features` files.
    """
    experiment_dir = Path(experiment_dir)
    if not experiment_dir.exists():
        raise PipelineError(f"Experiment folder does not exist: {experiment_dir}")

    timepoint_dirs = sorted(
        [p for p in experiment_dir.iterdir() if p.is_dir()],
        key=lambda p: natural_sort_key(p.name),
    )
    if not timepoint_dirs:
        raise PipelineError(f"No timepoint subfolders found in: {experiment_dir}")

    discovered: list[TimepointPaths] = []

    for tp_dir in timepoint_dirs:
        raw_path = tp_dir / raw_filename
        if not raw_path.exists():
            raise PipelineError(f"Raw volume not found: {raw_path}")

        seg_path = discover_segmentation_file(
            tp_dir,
            raw_filename=raw_filename,
            features_dirname=features_dirname,
            segmentation_filename=segmentation_filename,
        )

        features_dir = tp_dir / features_dirname
        if not features_dir.exists() or not features_dir.is_dir():
            raise PipelineError(f"Feature folder not found: {features_dir}")

        feature_paths = list_tiff_files(features_dir)
        if len(feature_paths) < n_patch_features:
            raise PipelineError(
                f"Timepoint {tp_dir.name} only has {len(feature_paths)} features, "
                f"but n_patch_features={n_patch_features}."
            )

        used_feature_paths = feature_paths[:n_patch_features]
        ignored_feature_paths = feature_paths[n_patch_features:]

        discovered.append(
            TimepointPaths(
                name=tp_dir.name,
                folder=tp_dir,
                raw_path=raw_path,
                segmentation_path=seg_path,
                feature_paths=used_feature_paths,
                ignored_feature_paths=ignored_feature_paths,
            )
        )

    return discovered


# -----------------------------------------------------------------------------
# Segmentation preparation
# -----------------------------------------------------------------------------


def extract_label_geometries(segmentation_yxz: np.ndarray) -> dict[int, LabelGeometry]:
    """Extract voxel coordinates, centroids, volumes, and cropped masks for all labels."""
    flat = segmentation_yxz.ravel()
    nz_idx = np.flatnonzero(flat != 0)
    if nz_idx.size == 0:
        return {}

    labels = flat[nz_idx].astype(np.int64, copy=False)
    order = np.argsort(labels, kind="mergesort")
    nz_idx = nz_idx[order]
    labels = labels[order]

    unique_labels, starts = np.unique(labels, return_index=True)
    counts = np.diff(np.r_[starts, len(labels)])

    coords_all = np.column_stack(np.unravel_index(nz_idx, segmentation_yxz.shape)).astype(
        np.int32, copy=False
    )

    geometries: dict[int, LabelGeometry] = {}
    for label_id, start, count in zip(unique_labels.tolist(), starts.tolist(), counts.tolist()):
        coords = coords_all[start : start + count]
        centroid = coords.mean(axis=0, dtype=np.float64)
        bbox_start = coords.min(axis=0)
        bbox_stop = coords.max(axis=0) + 1
        local_shape = tuple((bbox_stop - bbox_start).tolist())
        local_coords = coords - bbox_start

        local_mask = np.zeros(local_shape, dtype=bool)
        local_mask[tuple(local_coords.T)] = True

        geometries[int(label_id)] = LabelGeometry(
            label_id=int(label_id),
            vox_coords=coords,
            centroid=centroid,
            volume=int(coords.shape[0]),
            bbox_start=bbox_start.astype(np.int32, copy=False),
            bbox_stop=bbox_stop.astype(np.int32, copy=False),
            local_mask=local_mask,
            local_centroid=centroid - bbox_start,
        )

    return geometries


def prepare_timepoint(
    index: int,
    paths: TimepointPaths,
    *,
    tiff_axes: str = "zyx",
) -> PreparedTimepoint:
    seg = read_tiff_volume(paths.segmentation_path, axes=tiff_axes)
    geometries = extract_label_geometries(seg)

    label_ids = np.array(sorted(geometries.keys()), dtype=np.int64)
    if label_ids.size == 0:
        centroids = np.empty((0, 3), dtype=np.float64)
        volumes = np.empty((0,), dtype=np.int64)
    else:
        centroids = np.vstack([geometries[int(lbl)].centroid for lbl in label_ids])
        volumes = np.array([geometries[int(lbl)].volume for lbl in label_ids], dtype=np.int64)

    return PreparedTimepoint(
        index=index,
        paths=paths,
        shape_yxz=tuple(int(x) for x in seg.shape),
        geometries=geometries,
        label_ids=label_ids,
        centroids=centroids,
        volumes=volumes,
    )


# -----------------------------------------------------------------------------
# Pairwise geometry and feature metrics
# -----------------------------------------------------------------------------


def anisotropic_distance(a: np.ndarray, b: np.ndarray, zratio: float) -> float:
    dyx = a[:2] - b[:2]
    dz = a[2] - b[2]
    return float(np.sqrt(np.sum(dyx**2) + zratio * (dz**2)))


def find_spatial_candidates(
    ref_tp: PreparedTimepoint,
    cand_tp: PreparedTimepoint,
    *,
    spatial_radius: tuple[float, float, float] = (20.0, 20.0, 10.0),
    zratio: float = 2.5,
) -> dict[int, tuple[np.ndarray, np.ndarray]]:
    """
    Return ref_label -> (candidate_ids, distances) for candidates inside the centroid box.
    Distances use the anisotropic metric from the MATLAB code.
    """
    if cand_tp.label_ids.size == 0:
        return {
            int(ref_id): (
                np.empty((0,), dtype=np.int64),
                np.empty((0,), dtype=np.float32),
            )
            for ref_id in ref_tp.label_ids
        }

    radius = np.asarray(spatial_radius, dtype=np.float64)
    out: dict[int, tuple[np.ndarray, np.ndarray]] = {}

    for ref_id in ref_tp.label_ids.tolist():
        ref_centroid = ref_tp.geometries[int(ref_id)].centroid
        in_box = np.all(
            np.abs(cand_tp.centroids - ref_centroid[None, :]) <= radius[None, :],
            axis=1,
        )
        cand_ids = cand_tp.label_ids[in_box]
        dists = np.array(
            [
                anisotropic_distance(
                    ref_centroid,
                    cand_tp.geometries[int(cid)].centroid,
                    zratio,
                )
                for cid in cand_ids
            ],
            dtype=np.float32,
        )
        order = np.argsort(dists, kind="mergesort")
        out[int(ref_id)] = (cand_ids[order].astype(np.int64, copy=False), dists[order])

    return out


def compute_alignment_overlap(
    ref_geom: LabelGeometry,
    cand_geom: LabelGeometry,
) -> tuple[float, np.ndarray, np.ndarray, int]:
    """
    Compute Dice overlap after centroid alignment and return linear overlap indices
    into ref/candidate cropped subvolumes.
    """
    ref_shape = np.asarray(ref_geom.local_mask.shape, dtype=np.int32)
    cand_shape = np.asarray(cand_geom.local_mask.shape, dtype=np.int32)

    if np.any(ref_shape < 2) or np.any(cand_shape < 2):
        return 0.0, np.empty((0,), dtype=np.int64), np.empty((0,), dtype=np.int64), 0

    offset = np.rint(ref_geom.local_centroid - cand_geom.local_centroid).astype(np.int32)

    ref_start = np.maximum(0, -offset)
    cand_start = np.maximum(0, offset)
    ref_end = ref_start + ref_shape
    cand_end = cand_start + cand_shape

    overlap_start = np.maximum(ref_start, cand_start)
    overlap_end = np.minimum(ref_end, cand_end)

    if np.any(overlap_end <= overlap_start):
        return 0.0, np.empty((0,), dtype=np.int64), np.empty((0,), dtype=np.int64), 0

    ref_slices = tuple(
        slice(int(overlap_start[d] - ref_start[d]), int(overlap_end[d] - ref_start[d]))
        for d in range(3)
    )
    cand_slices = tuple(
        slice(int(overlap_start[d] - cand_start[d]), int(overlap_end[d] - cand_start[d]))
        for d in range(3)
    )

    ref_region = ref_geom.local_mask[ref_slices]
    cand_region = cand_geom.local_mask[cand_slices]
    overlap_mask = ref_region & cand_region

    intersection = int(overlap_mask.sum())
    union_count = ref_geom.volume + cand_geom.volume
    dice = 0.0 if union_count == 0 else float((2.0 * intersection) / union_count)

    if intersection <= 1:
        return dice, np.empty((0,), dtype=np.int64), np.empty((0,), dtype=np.int64), intersection

    overlap_local = np.argwhere(overlap_mask).astype(np.int32, copy=False)
    ref_offsets = np.array([s.start for s in ref_slices], dtype=np.int32)
    cand_offsets = np.array([s.start for s in cand_slices], dtype=np.int32)

    ref_coords = overlap_local + ref_offsets[None, :]
    cand_coords = overlap_local + cand_offsets[None, :]

    ref_lin = np.ravel_multi_index(ref_coords.T, ref_geom.local_mask.shape)
    cand_lin = np.ravel_multi_index(cand_coords.T, cand_geom.local_mask.shape)

    return (
        dice,
        ref_lin.astype(np.int64, copy=False),
        cand_lin.astype(np.int64, copy=False),
        intersection,
    )


def compute_pair_metrics(
    ref_tp: PreparedTimepoint,
    cand_tp: PreparedTimepoint,
    *,
    tiff_axes: str = "zyx",
    n_features: int = 384,
    spatial_radius: tuple[float, float, float] = (20.0, 20.0, 10.0),
    zratio: float = 2.5,
) -> dict[int, RefCandidateMetrics]:
    """
    Compute all sparse feature metrics for one consecutive pair of timepoints.

    This is a corrected version of script 2:
    - candidate IDs are always real label IDs, never row indices;
    - Dice is computed once per pair (it does not depend on feature channel);
    - correlation / MSE are stored per feature and per candidate, which allows
      MATLAB-like recomputation of vote tallies after removing assigned pairs.
    """
    spatial = find_spatial_candidates(
        ref_tp,
        cand_tp,
        spatial_radius=spatial_radius,
        zratio=zratio,
    )

    metrics_by_ref: dict[int, RefCandidateMetrics] = {}
    overlap_cache: dict[int, list[tuple[np.ndarray, np.ndarray]]] = {}

    # Precompute geometry-only overlaps once.
    for ref_id in ref_tp.label_ids.tolist():
        cand_ids, distances = spatial[int(ref_id)]
        n_cand = len(cand_ids)
        dice = np.full((n_cand,), np.nan, dtype=np.float32)
        overlap_counts = np.zeros((n_cand,), dtype=np.int32)
        pair_cache: list[tuple[np.ndarray, np.ndarray]] = []

        ref_geom = ref_tp.geometries[int(ref_id)]
        for j, cand_id in enumerate(cand_ids.tolist()):
            cand_geom = cand_tp.geometries[int(cand_id)]
            dice_j, ref_lin, cand_lin, overlap_count = compute_alignment_overlap(
                ref_geom, cand_geom
            )
            dice[j] = np.float32(dice_j)
            overlap_counts[j] = overlap_count
            pair_cache.append((ref_lin, cand_lin))

        corr = np.full((n_features, n_cand), np.nan, dtype=np.float32)
        mse = np.full((n_features, n_cand), np.nan, dtype=np.float32)

        metrics_by_ref[int(ref_id)] = RefCandidateMetrics(
            ref_label=int(ref_id),
            candidate_ids=cand_ids,
            distances=distances.astype(np.float32, copy=False),
            dice=dice,
            overlap_counts=overlap_counts,
            corr=corr,
            mse=mse,
        )
        overlap_cache[int(ref_id)] = pair_cache

    # Feature loop.
    for f in range(n_features):
        ref_vol = read_tiff_volume(ref_tp.paths.feature_paths[f], axes=tiff_axes).astype(
            np.float32, copy=False
        )
        cand_vol = read_tiff_volume(cand_tp.paths.feature_paths[f], axes=tiff_axes).astype(
            np.float32, copy=False
        )

        if ref_vol.shape != ref_tp.shape_yxz:
            raise PipelineError(
                f"Feature {ref_tp.paths.feature_paths[f]} has shape {ref_vol.shape}, "
                f"expected {ref_tp.shape_yxz}."
            )
        if cand_vol.shape != cand_tp.shape_yxz:
            raise PipelineError(
                f"Feature {cand_tp.paths.feature_paths[f]} has shape {cand_vol.shape}, "
                f"expected {cand_tp.shape_yxz}."
            )

        ref_cache: dict[int, np.ndarray] = {}
        cand_cache: dict[int, np.ndarray] = {}

        for ref_id, ref_metrics in metrics_by_ref.items():
            if ref_metrics.candidate_ids.size == 0:
                continue

            ref_geom = ref_tp.geometries[int(ref_id)]
            ref_flat = ref_cache.get(ref_id)
            if ref_flat is None:
                ref_flat = ref_vol[ref_geom.bbox_slices].ravel()
                ref_cache[ref_id] = ref_flat

            for j, cand_id in enumerate(ref_metrics.candidate_ids.tolist()):
                if ref_metrics.overlap_counts[j] <= 1:
                    continue

                ref_lin, cand_lin = overlap_cache[ref_id][j]
                if ref_lin.size <= 1:
                    continue

                cand_flat = cand_cache.get(int(cand_id))
                if cand_flat is None:
                    cand_geom = cand_tp.geometries[int(cand_id)]
                    cand_flat = cand_vol[cand_geom.bbox_slices].ravel()
                    cand_cache[int(cand_id)] = cand_flat

                ref_vals = ref_flat[ref_lin]
                cand_vals = cand_flat[cand_lin]

                diff = ref_vals - cand_vals
                ref_metrics.mse[f, j] = float(np.mean(diff * diff))

                ref_std = float(np.std(ref_vals))
                cand_std = float(np.std(cand_vals))
                if ref_std > 0.0 and cand_std > 0.0:
                    ref_metrics.corr[f, j] = float(np.corrcoef(ref_vals, cand_vals)[0, 1])

    return metrics_by_ref


# -----------------------------------------------------------------------------
# Vote summaries and assignment logic (cleaned-up script 2)
# -----------------------------------------------------------------------------


def build_summary_from_metrics(
    metrics_by_ref: dict[int, RefCandidateMetrics],
    *,
    available_candidates: dict[int, set[int]] | None = None,
    dice_threshold: float = 0.5,
    corr_threshold: float = 0.5,
) -> list[RefSummary]:
    summaries: list[RefSummary] = []

    for ref_id in sorted(metrics_by_ref.keys()):
        metrics = metrics_by_ref[ref_id]
        if available_candidates is None:
            allowed = np.ones(metrics.candidate_ids.shape[0], dtype=bool)
        else:
            allowed_ids = available_candidates.get(ref_id, set())
            allowed = np.array(
                [int(cid) in allowed_ids for cid in metrics.candidate_ids.tolist()],
                dtype=bool,
            )

        candidates: dict[int, CandidateVote] = {}
        for cid, dist, dice in zip(
            metrics.candidate_ids.tolist(),
            metrics.distances.tolist(),
            metrics.dice.tolist(),
        ):
            if (
                available_candidates is not None
                and int(cid) not in available_candidates.get(ref_id, set())
            ):
                continue
            candidates[int(cid)] = CandidateVote(
                candidate_label=int(cid),
                wins=0,
                features=[],
                distance=float(dist),
                dice=float(dice),
            )

        if np.any(allowed):
            for f in range(metrics.corr.shape[0]):
                good = (
                    allowed
                    & np.isfinite(metrics.dice)
                    & (metrics.dice > dice_threshold)
                    & np.isfinite(metrics.corr[f])
                    & (metrics.corr[f] > corr_threshold)
                    & np.isfinite(metrics.mse[f])
                )
                if not np.any(good):
                    continue
                good_idx = np.where(good)[0]
                chosen_j = int(good_idx[np.argmin(metrics.mse[f, good_idx])])
                chosen_cid = int(metrics.candidate_ids[chosen_j])
                candidates[chosen_cid].wins += 1
                candidates[chosen_cid].features.append(f)

        ordered = sorted(
            candidates.values(),
            key=lambda cv: (
                -cv.wins,
                np.inf if cv.distance is None else cv.distance,
                cv.candidate_label,
            ),
        )
        summaries.append(RefSummary(ref_label=ref_id, candidates=ordered))

    return summaries


def summary_to_wide_rows(summary: list[RefSummary]) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    max_cands = max((len(item.candidates) for item in summary), default=0)
    for item in summary:
        row: dict[str, Any] = {"RefLabel": item.ref_label}
        for k, cand in enumerate(item.candidates, start=1):
            row[f"Cand{k}"] = cand.candidate_label
            row[f"Wins{k}"] = cand.wins
            row[f"Features{k}"] = ",".join(str(f) for f in cand.features)
            row[f"Distance{k}"] = cand.distance
            row[f"Dice{k}"] = cand.dice
        for k in range(len(item.candidates) + 1, max_cands + 1):
            row[f"Cand{k}"] = None
            row[f"Wins{k}"] = None
            row[f"Features{k}"] = ""
            row[f"Distance{k}"] = None
            row[f"Dice{k}"] = None
        rows.append(row)
    return rows


def _remove_assigned_from_candidate_pool(
    candidate_pool: dict[int, set[int]],
    *,
    assigned_refs: Iterable[int],
    assigned_cands: Iterable[int],
) -> dict[int, set[int]]:
    assigned_refs = set(int(x) for x in assigned_refs)
    assigned_cands = set(int(x) for x in assigned_cands)

    out: dict[int, set[int]] = {}
    for ref_id, cands in candidate_pool.items():
        if ref_id in assigned_refs:
            continue
        out[ref_id] = set(int(c) for c in cands if int(c) not in assigned_cands)
    return out


def run_assignment_logic(
    metrics_by_ref: dict[int, RefCandidateMetrics],
    *,
    min_distance_to_remove_cand: float = 3.0,
    vote_thresholds: tuple[int, ...] = (320, 300, 280, 260),
    dice_threshold: float = 0.5,
    corr_threshold: float = 0.5,
) -> tuple[list[AssignmentRecord], list[RefSummary], list[list[RefSummary]], list[dict[str, float]]]:
    """Resolve pairs using distance prefilter, vote thresholds, then global closest."""
    all_ref_ids = sorted(metrics_by_ref.keys())
    candidate_pool: dict[int, set[int]] = {
        ref_id: set(metrics.candidate_ids.tolist()) for ref_id, metrics in metrics_by_ref.items()
    }

    initial_summary = build_summary_from_metrics(
        metrics_by_ref,
        available_candidates=candidate_pool,
        dice_threshold=dice_threshold,
        corr_threshold=corr_threshold,
    )
    summary_history: list[list[RefSummary]] = [initial_summary]
    assignments: list[AssignmentRecord] = []
    summary_distance_prefilter: list[dict[str, float]] = []

    # ------------------------------------------------------------------
    # Step 0: nearest-neighbor prefilter below a distance threshold.
    # We do this globally by sorting all ref->nearest pairs by distance,
    # then greedily taking non-conflicting assignments.
    # ------------------------------------------------------------------
    nearest_pairs: list[tuple[float, int, int]] = []
    for ref_id in all_ref_ids:
        metrics = metrics_by_ref[ref_id]
        if metrics.candidate_ids.size == 0:
            continue
        best_j = int(np.argmin(metrics.distances))
        min_dist = float(metrics.distances[best_j])
        if min_dist < min_distance_to_remove_cand:
            nearest_pairs.append((min_dist, ref_id, int(metrics.candidate_ids[best_j])))
        else:
            summary_distance_prefilter.append(
                {"RefLabel": float(ref_id), "MinDistance": min_dist}
            )

    assigned_refs: set[int] = set()
    assigned_cands: set[int] = set()
    for dist, ref_id, cand_id in sorted(nearest_pairs, key=lambda x: (x[0], x[1], x[2])):
        if ref_id in assigned_refs or cand_id in assigned_cands:
            continue
        assignments.append(
            AssignmentRecord(
                ref_label=ref_id,
                candidate_label=cand_id,
                method="distance_prefilter",
                distance=dist,
                wins=None,
            )
        )
        assigned_refs.add(ref_id)
        assigned_cands.add(cand_id)

    candidate_pool = _remove_assigned_from_candidate_pool(
        candidate_pool,
        assigned_refs=assigned_refs,
        assigned_cands=assigned_cands,
    )

    summary_current = build_summary_from_metrics(
        metrics_by_ref,
        available_candidates=candidate_pool,
        dice_threshold=dice_threshold,
        corr_threshold=corr_threshold,
    )
    summary_history.append(summary_current)

    # ------------------------------------------------------------------
    # Step 1: iterative high-vote unique matches.
    # ------------------------------------------------------------------
    used_vote_thresholds = tuple(
        sorted(set(int(t) for t in vote_thresholds if int(t) > 0), reverse=True)
    )

    for threshold in used_vote_thresholds:
        while True:
            if not summary_current:
                break

            top_candidates: list[tuple[int, int, int]] = []
            for item in summary_current:
                if not item.candidates:
                    continue
                top = item.candidates[0]
                if top.wins > threshold:
                    top_candidates.append((item.ref_label, top.candidate_label, top.wins))

            if not top_candidates:
                break

            counts = defaultdict(int)
            for _, cand_id, _ in top_candidates:
                counts[cand_id] += 1

            accepted: list[tuple[int, int, int]] = [
                (ref_id, cand_id, wins)
                for ref_id, cand_id, wins in top_candidates
                if counts[cand_id] == 1
            ]
            if not accepted:
                break

            new_refs: set[int] = set()
            new_cands: set[int] = set()
            for ref_id, cand_id, wins in accepted:
                if ref_id in new_refs or cand_id in new_cands:
                    continue
                assignments.append(
                    AssignmentRecord(
                        ref_label=ref_id,
                        candidate_label=cand_id,
                        method=f"vote_threshold_{threshold}",
                        distance=None,
                        wins=wins,
                    )
                )
                new_refs.add(ref_id)
                new_cands.add(cand_id)

            if not new_refs:
                break

            candidate_pool = _remove_assigned_from_candidate_pool(
                candidate_pool,
                assigned_refs=new_refs,
                assigned_cands=new_cands,
            )
            summary_current = build_summary_from_metrics(
                metrics_by_ref,
                available_candidates=candidate_pool,
                dice_threshold=dice_threshold,
                corr_threshold=corr_threshold,
            )
            summary_history.append(summary_current)

    # ------------------------------------------------------------------
    # Step 2: greedy global closest for leftovers.
    # ------------------------------------------------------------------
    remaining_pairs: list[tuple[float, int, int]] = []
    for ref_id, remaining_cands in candidate_pool.items():
        if not remaining_cands:
            continue
        metrics = metrics_by_ref[ref_id]
        for j, cand_id in enumerate(metrics.candidate_ids.tolist()):
            if int(cand_id) in remaining_cands:
                remaining_pairs.append((float(metrics.distances[j]), ref_id, int(cand_id)))

    used_refs = {a.ref_label for a in assignments}
    used_cands = {a.candidate_label for a in assignments}
    for dist, ref_id, cand_id in sorted(remaining_pairs, key=lambda x: (x[0], x[1], x[2])):
        if ref_id in used_refs or cand_id in used_cands:
            continue
        assignments.append(
            AssignmentRecord(
                ref_label=ref_id,
                candidate_label=cand_id,
                method="global_closest",
                distance=dist,
                wins=None,
            )
        )
        used_refs.add(ref_id)
        used_cands.add(cand_id)

    candidate_pool = _remove_assigned_from_candidate_pool(
        candidate_pool,
        assigned_refs=used_refs,
        assigned_cands=used_cands,
    )
    summary_current = build_summary_from_metrics(
        metrics_by_ref,
        available_candidates=candidate_pool,
        dice_threshold=dice_threshold,
        corr_threshold=corr_threshold,
    )
    summary_history.append(summary_current)

    assignments.sort(key=lambda a: (a.ref_label, a.candidate_label))
    return assignments, initial_summary, summary_history, summary_distance_prefilter


# -----------------------------------------------------------------------------
# Tracks and output volumes
# -----------------------------------------------------------------------------


def build_tracks(
    prepared_timepoints: list[PreparedTimepoint],
    pair_results: list[PairResult],
) -> list[Track]:
    if not prepared_timepoints:
        return []

    tracks: list[Track] = []
    active: dict[int, int] = {}

    def start_track(timepoint_idx: int, label_id: int) -> int:
        geom = prepared_timepoints[timepoint_idx - 1].geometries[int(label_id)]
        track_id = len(tracks) + 1
        track = Track(
            track_id=track_id,
            points=[
                TrackPoint(
                    timepoint=timepoint_idx,
                    label_id=int(label_id),
                    centroid=tuple(float(x) for x in geom.centroid.tolist()),
                    volume=int(geom.volume),
                )
            ],
            start_time=timepoint_idx,
            length=1,
        )
        tracks.append(track)
        return track_id

    def ensure_track_point(track_id: int, timepoint_idx: int, label_id: int) -> None:
        track = tracks[track_id - 1]
        if any((p.timepoint == timepoint_idx and p.label_id == label_id) for p in track.points):
            return
        geom = prepared_timepoints[timepoint_idx - 1].geometries[int(label_id)]
        track.points.append(
            TrackPoint(
                timepoint=timepoint_idx,
                label_id=int(label_id),
                centroid=tuple(float(x) for x in geom.centroid.tolist()),
                volume=int(geom.volume),
            )
        )

    if prepared_timepoints[0].label_ids.size > 0:
        for label_id in prepared_timepoints[0].label_ids.tolist():
            tid = start_track(1, int(label_id))
            active[int(label_id)] = tid

    for pair_idx, pair_result in enumerate(pair_results, start=1):
        current_tp = prepared_timepoints[pair_idx - 1]
        next_tp = prepared_timepoints[pair_idx]

        for label_id in current_tp.label_ids.tolist():
            if int(label_id) not in active:
                active[int(label_id)] = start_track(pair_idx, int(label_id))

        next_active: dict[int, int] = {}
        for ref_label, cand_label in sorted(pair_result.final_assignment.items()):
            if int(ref_label) not in active:
                active[int(ref_label)] = start_track(pair_idx, int(ref_label))
            track_id = active[int(ref_label)]
            ensure_track_point(track_id, pair_idx, int(ref_label))
            ensure_track_point(track_id, pair_idx + 1, int(cand_label))
            next_active[int(cand_label)] = track_id

        active = next_active

        if pair_idx == len(pair_results):
            existing_last_frame = {
                p.label_id
                for tr in tracks
                for p in tr.points
                if p.timepoint == pair_idx + 1
            }
            for label_id in next_tp.label_ids.tolist():
                if int(label_id) not in existing_last_frame:
                    start_track(pair_idx + 1, int(label_id))

    for track in tracks:
        track.points.sort(key=lambda p: p.timepoint)
        track.start_time = min(p.timepoint for p in track.points)
        track.length = len(track.points)

    tracks.sort(key=lambda tr: tr.track_id)
    return tracks


def save_track_volumes(
    prepared_timepoints: list[PreparedTimepoint],
    tracks: list[Track],
    output_dir: str | Path,
    *,
    tiff_axes: str = "zyx",
    prefix: str = "Tracks",
) -> None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if not prepared_timepoints:
        return

    max_track_id = max((tr.track_id for tr in tracks), default=0)
    if max_track_id <= np.iinfo(np.uint16).max:
        out_dtype = np.uint16
    elif max_track_id <= np.iinfo(np.uint32).max:
        out_dtype = np.uint32
    else:
        out_dtype = np.uint64

    track_lookup: dict[int, list[TrackPoint]] = {tr.track_id: tr.points for tr in tracks}

    by_time: dict[int, list[tuple[int, int]]] = defaultdict(list)
    for track_id, points in track_lookup.items():
        for point in points:
            by_time[point.timepoint].append((track_id, point.label_id))

    for tp in prepared_timepoints:
        vol = np.zeros(tp.shape_yxz, dtype=out_dtype)
        for track_id, label_id in by_time.get(tp.index, []):
            geom = tp.geometries[int(label_id)]
            coords = geom.vox_coords
            vol[coords[:, 0], coords[:, 1], coords[:, 2]] = track_id
        out_path = output_dir / f"{prefix}_t{tp.index:03d}.tif"
        write_tiff_volume(out_path, vol, axes=tiff_axes)


# -----------------------------------------------------------------------------
# Serialization helpers
# -----------------------------------------------------------------------------


def save_pairwise_assignments_csv(pair_results: list[PairResult], output_dir: str | Path) -> None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for result in pair_results:
        path = output_dir / f"pairwise_assignments_t{result.t_ref:03d}_to_t{result.t_cand:03d}.csv"
        with path.open("w", newline="") as f:
            writer = csv.DictWriter(
                f,
                fieldnames=["RefLabel", "AssignedCandidate", "Method", "Distance", "Wins"],
            )
            writer.writeheader()
            for a in sorted(result.assignments, key=lambda x: (x.ref_label, x.candidate_label)):
                writer.writerow(
                    {
                        "RefLabel": a.ref_label,
                        "AssignedCandidate": a.candidate_label,
                        "Method": a.method,
                        "Distance": a.distance,
                        "Wins": a.wins,
                    }
                )


def save_tracks_csv(tracks: list[Track], output_dir: str | Path) -> None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    max_time = max((p.timepoint for tr in tracks for p in tr.points), default=0)
    fieldnames = ["TrackID", "StartTime", "Length"] + [f"Label_t{t}" for t in range(1, max_time + 1)]
    path = output_dir / "tracks.csv"
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for tr in tracks:
            row: dict[str, Any] = {
                "TrackID": tr.track_id,
                "StartTime": tr.start_time,
                "Length": tr.length,
            }
            time_to_label = {p.timepoint: p.label_id for p in tr.points}
            for t in range(1, max_time + 1):
                row[f"Label_t{t}"] = time_to_label.get(t)
            writer.writerow(row)


def save_pickles(
    prepared_timepoints: list[PreparedTimepoint],
    pair_results: list[PairResult],
    tracks: list[Track],
    output_dir: str | Path,
) -> None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    with (output_dir / "pair_results.pkl").open("wb") as f:
        pickle.dump(pair_results, f)

    with (output_dir / "tracks.pkl").open("wb") as f:
        pickle.dump(tracks, f)

    minimal_timepoints = [
        {
            "index": tp.index,
            "name": tp.paths.name,
            "shape_yxz": tp.shape_yxz,
            "n_labels": int(len(tp.geometries)),
        }
        for tp in prepared_timepoints
    ]
    with (output_dir / "experiment_summary.pkl").open("wb") as f:
        pickle.dump(minimal_timepoints, f)


# -----------------------------------------------------------------------------
# Full pipeline
# -----------------------------------------------------------------------------


def run_pipeline(
    experiment_dir: str | Path,
    output_dir: str | Path,
    *,
    raw_filename: str = "volume_unnorm.tif",
    segmentation_filename: str | None = None,
    features_dirname: str = "features_tif",
    n_patch_features: int = 384,
    tiff_axes: str = "zyx",
    spatial_radius: tuple[float, float, float] = (20.0, 20.0, 10.0),
    zratio: float = 2.5,
    min_distance_to_remove_cand: float = 3.0,
    vote_thresholds: tuple[int, ...] | None = None,
    dice_threshold: float = 0.5,
    corr_threshold: float = 0.5,
    save_pickles_flag: bool = True,
    save_csv_flag: bool = True,
    save_track_volumes_flag: bool = True,
) -> tuple[list[PreparedTimepoint], list[PairResult], list[Track]]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    discovered = discover_experiment(
        experiment_dir,
        raw_filename=raw_filename,
        segmentation_filename=segmentation_filename,
        features_dirname=features_dirname,
        n_patch_features=n_patch_features,
    )

    prepared_timepoints = [
        prepare_timepoint(i + 1, tp, tiff_axes=tiff_axes) for i, tp in enumerate(discovered)
    ]

    for tp in prepared_timepoints:
        raw_shape = read_tiff_volume(tp.paths.raw_path, axes=tiff_axes).shape
        seg_shape = read_tiff_volume(tp.paths.segmentation_path, axes=tiff_axes).shape
        if raw_shape != tp.shape_yxz or seg_shape != tp.shape_yxz:
            raise PipelineError(
                f"Shape mismatch in timepoint {tp.paths.name}: raw={raw_shape}, seg={seg_shape}, expected={tp.shape_yxz}."
            )

    if vote_thresholds is None:
        vote_thresholds = tuple(th for th in (400, 320, 300, 280, 260) if th <= n_patch_features)
        if not vote_thresholds:
            vote_thresholds = (max(1, int(round(0.8 * n_patch_features))),)

    pair_results: list[PairResult] = []
    for i in range(len(prepared_timepoints) - 1):
        ref_tp = prepared_timepoints[i]
        cand_tp = prepared_timepoints[i + 1]

        metrics_by_ref = compute_pair_metrics(
            ref_tp,
            cand_tp,
            tiff_axes=tiff_axes,
            n_features=n_patch_features,
            spatial_radius=spatial_radius,
            zratio=zratio,
        )

        assignments, initial_summary, summary_history, summary_distance_prefilter = run_assignment_logic(
            metrics_by_ref,
            min_distance_to_remove_cand=min_distance_to_remove_cand,
            vote_thresholds=vote_thresholds,
            dice_threshold=dice_threshold,
            corr_threshold=corr_threshold,
        )

        final_assignment = {a.ref_label: a.candidate_label for a in assignments}
        summary_current = summary_history[-1] if summary_history else []

        pair_results.append(
            PairResult(
                t_ref=ref_tp.index,
                t_cand=cand_tp.index,
                initial_summary=initial_summary,
                summary_current=summary_current,
                summary_history=summary_history,
                assignments=assignments,
                final_assignment=final_assignment,
                summary_distance_prefilter=summary_distance_prefilter,
            )
        )

    tracks = build_tracks(prepared_timepoints, pair_results)

    if save_track_volumes_flag:
        save_track_volumes(prepared_timepoints, tracks, output_dir, tiff_axes=tiff_axes)
    if save_csv_flag:
        save_pairwise_assignments_csv(pair_results, output_dir)
        save_tracks_csv(tracks, output_dir)
    if save_pickles_flag:
        save_pickles(prepared_timepoints, pair_results, tracks, output_dir)

    return prepared_timepoints, pair_results, tracks


# -----------------------------------------------------------------------------
# CLI
# -----------------------------------------------------------------------------


def parse_thresholds(text: str | None) -> tuple[int, ...] | None:
    if text is None or text.strip() == "":
        return None
    return tuple(int(tok.strip()) for tok in text.split(",") if tok.strip())


def main() -> None:
    parser = argparse.ArgumentParser(description="Track 3D segmented objects across timepoints.")
    parser.add_argument("experiment_dir", type=Path, help="Experiment folder containing timepoint subfolders.")
    parser.add_argument("output_dir", type=Path, help="Where to write outputs.")
    parser.add_argument(
        "--raw-filename",
        type=str,
        default="volume_unnorm.tif",
        help="Filename of the raw volume in each timepoint folder.",
    )
    parser.add_argument(
        "--segmentation-filename",
        type=str,
        default=None,
        help="Filename of the segmentation TIFF in each timepoint folder. If omitted, the script tries to auto-detect it.",
    )
    parser.add_argument(
        "--features-dirname",
        type=str,
        default="features_tif",
        help="Name of the folder containing feature TIFFs.",
    )
    parser.add_argument(
        "--n-patch-features",
        type=int,
        default=384,
        help="How many feature TIFFs to use from the naturally sorted feature list.",
    )
    parser.add_argument(
        "--tiff-axes",
        choices=["zyx", "yxz"],
        default="zyx",
        help=(
            "Axis order returned by tifffile.imread for your stacks before internal conversion. "
            "Use 'zyx' for standard multi-page TIFF stacks and 'yxz' if your reader already yields MATLAB-like Y-X-Z arrays."
        ),
    )
    parser.add_argument(
        "--spatial-radius",
        type=float,
        nargs=3,
        default=(20.0, 20.0, 10.0),
        metavar=("DX", "DY", "DZ"),
        help="Half-widths of the centroid candidate box in internal Y-X-Z coordinates.",
    )
    parser.add_argument(
        "--zratio",
        type=float,
        default=2.5,
        help="Anisotropy factor applied to z in centroid distance calculations.",
    )
    parser.add_argument(
        "--min-distance-to-remove-cand",
        type=float,
        default=3.0,
        help="Immediate assignment threshold for the distance prefilter.",
    )
    parser.add_argument(
        "--vote-thresholds",
        type=str,
        default=None,
        help="Comma-separated vote thresholds, e.g. '320,300,280,260'.",
    )
    parser.add_argument("--dice-threshold", type=float, default=0.5)
    parser.add_argument("--corr-threshold", type=float, default=0.5)
    parser.add_argument("--no-pickles", action="store_true", help="Do not write pickle outputs.")
    parser.add_argument("--no-csv", action="store_true", help="Do not write CSV outputs.")
    parser.add_argument("--no-track-volumes", action="store_true", help="Do not write track TIFF stacks.")

    args = parser.parse_args()

    run_pipeline(
        args.experiment_dir,
        args.output_dir,
        raw_filename=args.raw_filename,
        segmentation_filename=args.segmentation_filename,
        features_dirname=args.features_dirname,
        n_patch_features=args.n_patch_features,
        tiff_axes=args.tiff_axes,
        spatial_radius=tuple(float(x) for x in args.spatial_radius),
        zratio=float(args.zratio),
        min_distance_to_remove_cand=float(args.min_distance_to_remove_cand),
        vote_thresholds=parse_thresholds(args.vote_thresholds),
        dice_threshold=float(args.dice_threshold),
        corr_threshold=float(args.corr_threshold),
        save_pickles_flag=not args.no_pickles,
        save_csv_flag=not args.no_csv,
        save_track_volumes_flag=not args.no_track_volumes,
    )


if __name__ == "__main__":
    main()